# Model Panel And Sequence Construction

This notebook implements:

1. **Step 4: Daily Feature Panel** with
   - `date, ticker, feature_1..feature_k, target_return`
   - `target_return = log(P_{t+1}/P_t)`
2. **Step 5: Sequence Construction**
   - lookback window = 60 trading days
   - `X` as rolling 60-day feature windows
   - `y` as next-day return
   - forward-walk folds for cross-validation

It expects the PIT-engineered full-history panel exported by `data.ipynb`.

In [2]:
from __future__ import annotations

from pathlib import Path
import pickle
import sys

import numpy as np
import pandas as pd

# Resolve team_t root regardless of where notebook is launched from.
CWD = Path.cwd().resolve()
if (CWD / "team_t").exists():
    TEAM_ROOT = CWD / "team_t"
elif CWD.name == "team_t":
    TEAM_ROOT = CWD
elif CWD.name == "notebooks" and CWD.parent.name == "team_t":
    TEAM_ROOT = CWD.parent
else:
    TEAM_ROOT = next((p for p in [CWD, *CWD.parents] if p.name == "team_t"), None)

if TEAM_ROOT is None:
    raise FileNotFoundError("Could not resolve team_t directory from current working directory")

if str(TEAM_ROOT) not in sys.path:
    sys.path.insert(0, str(TEAM_ROOT))

from src.feature_engineering import build_lstm_tensors, get_stage_feature_columns


## 1) Parameters And Paths

- `LOOKBACK=60` for LSTM sequences
- Expanding forward-walk train years + 1-year dev + optional 1-year test
- Uses full-history PIT panel by default

In [3]:
LOOKBACK = 40
FORWARD_MIN_TRAIN_YEARS = 5
FORWARD_VALID_YEARS = 1
FORWARD_TEST_YEARS = 1

USE_ZSCORE_FEATURES = True
MODEL_MAX_FEATURE_STAGE = 5

OUTPUT_DIR = TEAM_ROOT / "data" / "lstm_draft" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PIT_PANEL_CANDIDATES = [
    OUTPUT_DIR / "feature_panel_pit_normalized_full_history.parquet",
    OUTPUT_DIR / "lstm_feature_panel_full_history.parquet",
]

MODEL_PANEL_PARQUET = OUTPUT_DIR / "lstm_model_panel_full_history.parquet"
MODEL_PANEL_CSV = OUTPUT_DIR / "lstm_model_panel_full_history.csv"

FORWARD_WALK_TENSORS_PKL = OUTPUT_DIR / f"lstm_forward_walk_tensors_lookback_{LOOKBACK}.pkl"
FORWARD_WALK_SUMMARY_CSV = OUTPUT_DIR / f"lstm_forward_walk_tensors_lookback_{LOOKBACK}_summary.csv"

print(f"TEAM_ROOT={TEAM_ROOT}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")


TEAM_ROOT=/Users/assortedsphinx/Desktop/team_t
OUTPUT_DIR=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed


## 2) Load PIT Feature Panel

Loads the first existing candidate file and reports coverage.

In [4]:
pit_source = next((p for p in PIT_PANEL_CANDIDATES if p.exists()), None)
if pit_source is None:
    checked = "\n".join(str(p) for p in PIT_PANEL_CANDIDATES)
    raise FileNotFoundError(f"No PIT panel found. Checked:\n{checked}")

panel = pd.read_parquet(pit_source)
if "date" not in panel.columns or "ticker" not in panel.columns:
    raise KeyError("PIT panel must contain 'date' and 'ticker' columns")

panel["date"] = pd.to_datetime(panel["date"], errors="coerce")
panel["ticker"] = panel["ticker"].astype(str).str.upper().str.strip()
panel = panel.dropna(subset=["date", "ticker"]).sort_values(["ticker", "date"]).reset_index(drop=True)

print(f"[pit source] {pit_source}")
print(f"[pit panel] rows={len(panel):,} tickers={panel['ticker'].nunique():,}")
print(f"[pit panel] date_min={panel['date'].min()} date_max={panel['date'].max()}")
panel.head(3)


[pit source] /Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/feature_panel_pit_normalized_full_history.parquet
[pit panel] rows=119,368 tickers=10
[pit panel] date_min=1962-01-02 00:00:00 date_max=2024-11-05 00:00:00


,ticker,date,open,close,volume,adj_close,adj_factor,adj_open,adj_close_intraday,per_end_date,...,rolling_beta,roe_change_z,margin_change_z,leverage_change_z,eps_growth_z,book_value_growth_z,log_return_z,rolling_beta_z,volume_ratio_z,target_next_log_ret
0,AAPL,1980-12-12,28.75,28.75,2093900.0,0.099192,0.00345,0.099192,0.099192,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.053584
1,AAPL,1980-12-15,27.38,27.25,785200.0,0.094017,0.00345,0.094466,0.094017,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,-2.178756,NaN,NaN,-0.076227
2,AAPL,1980-12-16,25.37,25.25,472000.0,0.087117,0.00345,0.087531,0.087117,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,-2.129674,NaN,NaN,0.024258


## 3) Step 4: Build Daily Modeling Panel

Target definition:

`target_return = log(P_{t+1} / P_t)` grouped by ticker, using `adj_close`.

Feature selection:

- Uses z-scored PIT features when present (`*_z`)
- Falls back to raw features if z-scored columns are missing

In [ ]:
if "adj_close" not in panel.columns:
    raise KeyError("Expected 'adj_close' in PIT panel to build target_return")

panel_model = panel.copy()
panel_model["adj_close"] = pd.to_numeric(panel_model["adj_close"], errors="coerce")
panel_model.loc[panel_model["adj_close"] <= 0, "adj_close"] = np.nan

panel_model["target_return"] = panel_model.groupby("ticker")["adj_close"].transform(
    lambda s: np.log(s.shift(-1)) - np.log(s)
)

base_feature_candidates = get_stage_feature_columns(max_stage=MODEL_MAX_FEATURE_STAGE)
base_feature_candidates = [c for c in base_feature_candidates if c in panel_model.columns]
z_feature_candidates = [f"{c}_z" for c in base_feature_candidates]
legacy_feature_candidates = [
    "adj_close",
    "volume",
    "tot_debt_tot_equity",
    "ret_equity",
    "profit_margin",
    "book_val_per_share",
    "diluted_net_eps",
    "price_to_book",
    "beta_252d",
]

def _complete_row_count(df: pd.DataFrame, cols: list[str]) -> int:
    if not cols:
        return 0
    return int(df[cols].notna().all(axis=1).sum())

def _max_ticker_year_rows(df: pd.DataFrame, cols: list[str]) -> int:
    if not cols:
        return 0
    valid = df[cols + ["target_return"]].notna().all(axis=1)
    if not bool(valid.any()):
        return 0
    tmp = df.loc[valid, ["ticker", "date"]].copy()
    tmp["year"] = tmp["date"].dt.year
    counts = tmp.groupby(["year", "ticker"]).size()
    if counts.empty:
        return 0
    return int(counts.max())

candidate_sets: list[tuple[str, list[str]]] = []
if USE_ZSCORE_FEATURES:
    candidate_sets.append(("zscore", [c for c in z_feature_candidates if c in panel_model.columns]))
candidate_sets.append(("base", [c for c in base_feature_candidates if c in panel_model.columns]))
candidate_sets.append(("legacy", [c for c in legacy_feature_candidates if c in panel_model.columns]))

candidate_summary: list[dict[str, object]] = []
for name, cols in candidate_sets:
    candidate_summary.append(
        {
            "candidate": name,
            "feature_count": len(cols),
            "complete_rows": _complete_row_count(panel_model, cols),
            "max_ticker_year_rows": _max_ticker_year_rows(panel_model, cols),
            "feature_cols": cols,
        }
    )

candidate_df = pd.DataFrame(candidate_summary)
if candidate_df.empty:
    raise ValueError("No model feature columns found in PIT panel")

candidate_lookup = {row["candidate"]: row for _, row in candidate_df.iterrows()}
min_required_rows = LOOKBACK * max(int(panel_model["ticker"].nunique()), 1)

selection_order: list[str] = []
if USE_ZSCORE_FEATURES:
    selection_order.append("zscore")
selection_order.extend(["base", "legacy"])

selected = None
for name in selection_order:
    row = candidate_lookup.get(name)
    if row is None:
        continue
    feature_count = int(row["feature_count"])
    complete_rows = int(row["complete_rows"])
    max_ticker_year_rows = int(row["max_ticker_year_rows"])
    if feature_count == 0 or complete_rows == 0:
        continue
    if max_ticker_year_rows <= LOOKBACK:
        print(
            f"[feature selection] {name} set is too sparse within yearly folds "
            f"(max_ticker_year_rows={max_ticker_year_rows} <= lookback={LOOKBACK}); trying fallback"
        )
        continue
    if name == "zscore" and complete_rows < min_required_rows:
        print(
            "[feature selection] z-score set is too sparse for sequence construction "
            f"({complete_rows} < {min_required_rows}); falling back to base features"
        )
        continue
    selected = row
    break

if selected is None:
    candidate_df_sorted = candidate_df.sort_values(
        ["max_ticker_year_rows", "complete_rows", "feature_count"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    if int(candidate_df_sorted.loc[0, "feature_count"]) == 0:
        raise ValueError("No usable feature set available for model panel")
    selected = candidate_df_sorted.loc[0]
    print(
        "[feature selection] fallback to densest available set: "
        f"{selected['candidate']} (max_ticker_year_rows={int(selected['max_ticker_year_rows'])})"
    )

feature_cols = list(selected["feature_cols"])
feature_source = str(selected["candidate"])

if int(selected["complete_rows"]) < min_required_rows:
    print(
        "[feature selection] Warning: selected feature set has limited complete rows "
        f"({int(selected['complete_rows'])} < {min_required_rows})"
    )

panel_model = panel_model[["date", "ticker", *feature_cols, "target_return"]].copy()
panel_model = panel_model.dropna(subset=["target_return"]).sort_values(["ticker", "date"]).reset_index(drop=True)

dup_model = int(panel_model.duplicated(["ticker", "date"]).sum())
assert dup_model == 0, f"Found {dup_model} duplicate (ticker, date) rows in panel_model"

print("panel rows:", len(panel_model))
print("unique tickers:", panel_model["ticker"].nunique())
print("date range:", panel_model["date"].min(), panel_model["date"].max())


print(f"[model panel] feature_stage={MODEL_MAX_FEATURE_STAGE}")
print(f"[model panel] feature_source={feature_source}")
print(f"[model panel] feature_cols={feature_cols}")
print(f"[model panel] rows={len(panel_model):,} tickers={panel_model['ticker'].nunique():,}")
print(f"[model panel] date_min={panel_model['date'].min()} date_max={panel_model['date'].max()}")

missing_frac = panel_model[feature_cols + ["target_return"]].isna().mean().sort_values(ascending=False)
print("[model panel] missing fraction")
print(missing_frac.to_string())
print("[feature candidates]")
print(candidate_df[["candidate", "feature_count", "complete_rows", "max_ticker_year_rows"]].to_string(index=False))

panel_model.head(5)


## 4) Export Daily Modeling Panel

Writes Step 4 output for reuse in later experiments.

In [6]:
panel_model.to_parquet(MODEL_PANEL_PARQUET, index=False)
panel_model.to_csv(MODEL_PANEL_CSV, index=False)

print(f"[export panel] parquet={MODEL_PANEL_PARQUET}")
print(f"[export panel] csv={MODEL_PANEL_CSV}")

coverage = pd.DataFrame(
    [
        {
            "dataset": "Daily modeling panel",
            "rows": int(len(panel_model)),
            "tickers": int(panel_model["ticker"].nunique()),
            "date_min": panel_model["date"].min(),
            "date_max": panel_model["date"].max(),
        }
    ]
)
coverage


[export panel] parquet=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/lstm_model_panel_full_history.parquet
[export panel] csv=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/lstm_model_panel_full_history.csv


,dataset,rows,tickers,date_min,date_max
0,Daily modeling panel,119358,10,1962-01-02,2024-11-04


## 5) Step 5: Build 60-Day LSTM Sequences With Forward-Walk Folds

Fold design (expanding window):

- train: all years up to fold cutoff
- dev: next year
- test: following year when available

Each fold uses `build_lstm_tensors` from `src.feature_engineering`.

In [7]:
panel_fw = panel_model.copy()
panel_fw["date"] = pd.to_datetime(panel_fw["date"], errors="coerce")
panel_fw = panel_fw.dropna(subset=["date"]).sort_values(["ticker", "date"]).reset_index(drop=True)

years = sorted(panel_fw["date"].dt.year.dropna().unique().tolist())
min_needed = FORWARD_MIN_TRAIN_YEARS + FORWARD_VALID_YEARS
if len(years) < min_needed:
    raise ValueError(
        f"Not enough annual history for forward-walk folds: years={years}, need at least {min_needed}"
    )

forward_walk_tensors: dict[str, dict[str, object]] = {}
forward_walk_rows: list[dict[str, object]] = []
skipped_empty_folds = 0

for idx in range(FORWARD_MIN_TRAIN_YEARS, len(years) - FORWARD_VALID_YEARS + 1):
    train_years = years[:idx]
    dev_years = years[idx : idx + FORWARD_VALID_YEARS]
    test_years = years[
        idx + FORWARD_VALID_YEARS : idx + FORWARD_VALID_YEARS + FORWARD_TEST_YEARS
    ]

    train_start = pd.Timestamp(f"{train_years[0]}-01-01")
    train_end = pd.Timestamp(f"{train_years[-1]}-12-31")
    dev_start = pd.Timestamp(f"{dev_years[0]}-01-01")
    dev_end = pd.Timestamp(f"{dev_years[-1]}-12-31")

    fold_df = panel_fw.copy()
    conditions = [
        (fold_df["date"] >= train_start) & (fold_df["date"] <= train_end),
        (fold_df["date"] >= dev_start) & (fold_df["date"] <= dev_end),
    ]
    labels = ["train", "dev"]

    test_window = None
    if len(test_years) > 0:
        test_start = pd.Timestamp(f"{test_years[0]}-01-01")
        test_end = pd.Timestamp(f"{test_years[-1]}-12-31")
        conditions.append((fold_df["date"] >= test_start) & (fold_df["date"] <= test_end))
        labels.append("test")
        test_window = (str(test_start.date()), str(test_end.date()))

    fold_df["fw_split"] = np.select(conditions, labels, default="outside")

    fold_key = f"train_{train_years[0]}_{train_years[-1]}__dev_{dev_years[0]}_{dev_years[-1]}"
    if len(test_years) > 0:
        fold_key += f"__test_{test_years[0]}_{test_years[-1]}"

    tensors_fold = build_lstm_tensors(
        panel_df=fold_df,
        feature_cols=feature_cols,
        target_col="target_return",
        lookback=LOOKBACK,
        split_col="fw_split",
    )

    x_train, y_train = tensors_fold["train"]
    x_dev, y_dev = tensors_fold["dev"]
    x_test, y_test = tensors_fold["test"]

    # Skip folds without usable train/dev samples.
    if y_train.shape[0] == 0 or y_dev.shape[0] == 0:
        skipped_empty_folds += 1
        continue

    forward_walk_tensors[fold_key] = {
        "train_years": train_years,
        "dev_years": dev_years,
        "test_years": test_years,
        "train_window": (str(train_start.date()), str(train_end.date())),
        "dev_window": (str(dev_start.date()), str(dev_end.date())),
        "test_window": test_window,
        "tensors": tensors_fold,
    }

    forward_walk_rows.append(
        {
            "fold_key": fold_key,
            "train_window": f"{train_start.date()}:{train_end.date()}",
            "dev_window": f"{dev_start.date()}:{dev_end.date()}",
            "test_window": (
                f"{test_window[0]}:{test_window[1]}"
                if test_window is not None
                else "none"
            ),
            "train_samples": int(y_train.shape[0]),
            "dev_samples": int(y_dev.shape[0]),
            "test_samples": int(y_test.shape[0]),
            "lookback": int(LOOKBACK),
            "feature_count": int(len(feature_cols)),
        }
    )

summary_df = pd.DataFrame(forward_walk_rows)
print(f"Built {len(summary_df)} forward-walk folds (skipped_empty_folds={skipped_empty_folds})")
print("[fold coverage] tensor shapes by fold")
for fold_key, payload in forward_walk_tensors.items():
    xtr, ytr = payload["tensors"]["train"]
    xdv, ydv = payload["tensors"]["dev"]
    xte, yte = payload["tensors"]["test"]
    print(f"{fold_key}: train={xtr.shape}, dev={xdv.shape}, test={xte.shape}")

summary_df


Built 0 forward-walk folds (skipped_empty_folds=58)


""


## 6) Persist Sequence Outputs

Saves a pickle with all fold tensors and a CSV summary with fold/sample counts.

In [8]:
payload = {
    "lookback": LOOKBACK,
    "feature_cols": feature_cols,
    "target_col": "target_return",
    "panel_source": str(pit_source),
    "folds": forward_walk_tensors,
}

with open(FORWARD_WALK_TENSORS_PKL, "wb") as f:
    pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

summary_df.to_csv(FORWARD_WALK_SUMMARY_CSV, index=False)

print(f"[export tensors] {FORWARD_WALK_TENSORS_PKL}")
print(f"[export summary] {FORWARD_WALK_SUMMARY_CSV}")

if not summary_df.empty:
    print(summary_df.to_string(index=False))


[export tensors] /Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/lstm_forward_walk_tensors_lookback_60.pkl
[export summary] /Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/lstm_forward_walk_tensors_lookback_60_summary.csv


## 7) Sanity Check: Tensor Shapes

Validates the expected shape convention: `samples x lookback x features`.

In [9]:
if summary_df.empty:
    raise ValueError("No forward-walk folds were built")

non_empty = summary_df[summary_df["train_samples"] > 0]
if non_empty.empty:
    raise ValueError("All built folds have zero train samples")

sample_fold = non_empty.iloc[0]["fold_key"]
x_train, y_train = forward_walk_tensors[sample_fold]["tensors"]["train"]
x_dev, y_dev = forward_walk_tensors[sample_fold]["tensors"]["dev"]
x_test, y_test = forward_walk_tensors[sample_fold]["tensors"]["test"]

print(f"sample_fold={sample_fold}")
print(f"X_train shape={x_train.shape}")
print(f"y_train shape={y_train.shape}")
print(f"expected tensor shape = (samples, {LOOKBACK}, {len(feature_cols)})")

if x_train.ndim != 3:
    raise ValueError(f"Expected X_train.ndim==3, got {x_train.ndim}")
if x_train.shape[1] != LOOKBACK:
    raise ValueError(f"Expected lookback dimension {LOOKBACK}, got {x_train.shape[1]}")
if x_train.shape[2] != len(feature_cols):
    raise ValueError(f"Expected feature dimension {len(feature_cols)}, got {x_train.shape[2]}")

assert np.isfinite(x_train).all(), "X_train contains non-finite values"
assert np.isfinite(y_train).all(), "y_train contains non-finite values"

def _finite_check(split_name: str, x_arr: np.ndarray, y_arr: np.ndarray) -> None:
    if x_arr.shape[0] == 0 or y_arr.shape[0] == 0:
        print(f"[sanity] skip finite check for {split_name}: empty tensor")
        return
    assert np.isfinite(x_arr).all(), f"X_{split_name} contains non-finite values"
    assert np.isfinite(y_arr).all(), f"y_{split_name} contains non-finite values"
    print(f"[sanity] finite check passed for {split_name}")

_finite_check("dev", x_dev, y_dev)
_finite_check("test", x_test, y_test)

print("[sanity] tensor shape checks passed")


ValueError: No forward-walk folds were built